# 09 — File Upload, Validation & Processing

> **📓 Notebook · Module 06 · Intermediate**
> *Master the UPLOAD → VALIDATE → READ → CLEAN → ANALYZE → VISUALIZE → DOWNLOAD workflow.*

---

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

1. Use `st.file_uploader` to accept single and multiple files.
2. Validate uploaded files by type, size, and content.
3. Read CSV, Excel, and JSON files into Pandas DataFrames.
4. Detect and handle missing values, duplicates, and type issues.
5. Clean and transform data for analysis.
6. Provide download buttons for processed data.
7. Apply safe file handling practices.

## 📋 Prerequisites

- Completed [Notebook 07 — DataFrames, Tables & Pandas Integration](07_dataframes_tables_pandas.ipynb)
- Pandas basics (read_csv, DataFrame operations)
- Understanding of Streamlit widgets and session state

---

## 📚 Concept: The Upload Pipeline

```
UPLOAD → VALIDATE → READ → CLEAN → ANALYZE → VISUALIZE → DOWNLOAD
   │          │        │       │         │            │          │
   ▼          ▼        ▼       ▼         ▼            ▼          ▼
Widget   Check type  Parse   Handle    Charts,     Charts     CSV,
& size   & content   to DF   nulls,    tables,     & tables   Excel,
                      │     dtypes     metrics               JSON
                      ▼
                  Analysis-
                   ready DF
```

**The key insight:** Validation is not optional. Never trust uploaded data.

## 🧠 Intuition: Uploaded Files Are Strangers

Think of file uploads like accepting a package from a stranger:

1. **Check the label** — Is it the right type? (CSV, Excel, JSON)
2. **Weigh it** — Is it too large? (Size limits)
3. **Open carefully** — Use try/except (might be malformed)
4. **Inspect contents** — Are columns correct? Any missing values?
5. **Clean before use** — Remove duplicates, fix types, handle nulls
6. **Never execute** — Don't run code from uploaded files

---

## 🔧 Build It: Basic File Upload

Start with the simplest case: upload a CSV and display it.

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np

st.set_page_config(page_title="Notebook 09 — File Upload", layout="wide")

st.header("Basic CSV Upload")

uploaded_file = st.file_uploader(
    "Upload a CSV file",
    type=["csv"],
    help="Upload any CSV file to see it displayed as a DataFrame.",
)

if uploaded_file is not None:
    try:
        df = pd.read_csv(uploaded_file)
        st.success(f"✅ Loaded **{uploaded_file.name}** — {len(df):,} rows × {len(df.columns)} columns")
        st.dataframe(df, use_container_width=True, height=400)
    except Exception as e:
        st.error(f"Error reading file: {e}")
else:
    st.info("👆 Upload a CSV file to get started.")

---

## 🔧 Build It: Multi-Format Upload

Handle CSV, Excel, and JSON with one uploader.

In [ ]:
st.header("Multi-Format Upload")

uploaded = st.file_uploader(
    "Upload a data file",
    type=["csv", "xlsx", "xls", "json"],
    key="multi_format",
    help="Supported: CSV, Excel (.xlsx), JSON",
)

if uploaded is not None:
    suffix = uploaded.name.split(".")[-1].lower()
    st.write(f"**File:** {uploaded.name} | **Type:** {suffix.upper()} | **Size:** {uploaded.size / 1024:.1f} KB")

    try:
        if suffix == "csv":
            df = pd.read_csv(uploaded)
        elif suffix in ("xlsx", "xls"):
            # Show sheet selector for Excel
            xl = pd.ExcelFile(uploaded)
            if len(xl.sheet_names) > 1:
                sheet = st.selectbox("Select sheet", xl.sheet_names)
                df = pd.read_excel(xl, sheet_name=sheet)
            else:
                df = pd.read_excel(xl)
        elif suffix == "json":
            import json
            data = json.load(uploaded)
            if isinstance(data, list):
                df = pd.json_normalize(data)
            elif isinstance(data, dict):
                # Try common wrappers
                for key in ["data", "results", "records", "items"]:
                    if key in data:
                        data = data[key]
                        break
                df = pd.json_normalize(data if isinstance(data, list) else [data])
            else:
                st.error("Unrecognized JSON structure.")
                st.stop()
        else:
            st.error(f"Unsupported format: {suffix}")
            st.stop()

        st.success(f"✅ Loaded: {len(df):,} rows × {len(df.columns)} columns")
        st.dataframe(df.head(20), use_container_width=True, height=400)

    except Exception as e:
        st.error(f"Error reading {suffix.upper()} file: {e}")
else:
    st.info("👆 Upload a CSV, Excel, or JSON file.")

---

## 🔧 Build It: File Validation

Validate at three levels: type/size, content, and data quality.

In [ ]:
st.header("Three-Level Validation")

uploaded_val = st.file_uploader(
    "Upload CSV for validation",
    type=["csv"],
    max_upload_size=10,  # 10 MB limit
    key="validation",
)

if uploaded_val is not None:
    # Level 1: Size check
    size_mb = uploaded_val.size / (1024 * 1024)
    st.info(f"📊 **Level 1 — Size Check:** {size_mb:.2f} MB")

    # Level 2: Content check
    try:
        uploaded_val.seek(0)
        df_val = pd.read_csv(uploaded_val)
        st.success(f"✅ **Level 2 — Content Check:** Parsed successfully")
    except pd.errors.EmptyDataError:
        st.error("❌ File is empty.")
        st.stop()
    except pd.errors.ParserError:
        st.error("❌ Could not parse CSV. Check file format.")
        st.stop()

    # Level 3: Data quality check
    st.subheader("Level 3 — Data Quality")

    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Rows", f"{len(df_val):,}")
    col2.metric("Columns", len(df_val.columns))
    col3.metric("Null Cells", f"{df_val.isnull().sum().sum():,}")
    col4.metric("Duplicates", f"{df_val.duplicated().sum():,}")

    # Null summary
    null_info = pd.DataFrame({
        "Column": df_val.columns,
        "Dtype": df_val.dtypes.values,
        "Nulls": df_val.isnull().sum().values,
        "Null %": (df_val.isnull().mean() * 100).round(1).values,
    }).sort_values("Nulls", ascending=False)

    st.dataframe(null_info, hide_index=True, use_container_width=True)

---

## 🔧 Build It: Missing Value Handling

Let the user choose how to handle missing values.

In [ ]:
st.header("Missing Value Handling")

# Generate sample data with missing values for demo
np.random.seed(42)
sample_df = pd.DataFrame({
    "Name": ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank", "Grace", "Hank"],
    "Age": [25, np.nan, 35, 28, np.nan, 42, 31, np.nan],
    "Salary": [50000, 60000, np.nan, 55000, 70000, np.nan, 62000, 48000],
    "Department": ["Engineering", "Sales", "Engineering", None, "Sales", "Marketing", None, "Engineering"],
    "Rating": [4.5, 3.8, np.nan, 4.2, 4.8, 3.5, np.nan, 4.0],
})

st.write("**Original Data (with missing values):**")
st.dataframe(sample_df, use_container_width=True)

st.write(f"Missing values: **{sample_df.isnull().sum().sum()}** total")

# Strategy selector
strategy = st.selectbox(
    "Missing value strategy",
    ["Drop rows with any null", "Fill numeric with mean", "Fill numeric with median",
     "Fill categorical with mode", "Fill with custom value"],
    key="null_strategy",
)

df_clean = sample_df.copy()

if strategy == "Drop rows with any null":
    df_clean = df_clean.dropna()
elif strategy == "Fill numeric with mean":
    num_cols = df_clean.select_dtypes(include="number").columns
    df_clean[num_cols] = df_clean[num_cols].fillna(df_clean[num_cols].mean())
elif strategy == "Fill numeric with median":
    num_cols = df_clean.select_dtypes(include="number").columns
    df_clean[num_cols] = df_clean[num_cols].fillna(df_clean[num_cols].median())
elif strategy == "Fill categorical with mode":
    cat_cols = df_clean.select_dtypes(include="object").columns
    for col in cat_cols:
        if df_clean[col].isnull().any():
            df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
elif strategy == "Fill with custom value":
    fill_val = st.text_input("Fill value", "Unknown")
    df_clean = df_clean.fillna(fill_val)

st.write(f"**After '{strategy}':**")
st.dataframe(df_clean, use_container_width=True)
st.write(f"Rows: {len(sample_df)} → {len(df_clean)} | Nulls: {sample_df.isnull().sum().sum()} → {df_clean.isnull().sum().sum()}")

---

## 🔧 Build It: Data Cleaning Workflow

Apply multiple cleaning steps in sequence.

In [ ]:
st.header("Cleaning Workflow")

# Generate messy data
np.random.seed(42)
messy_df = pd.DataFrame({
    "  Name ": ["Alice", "bob", "Charlie", "Diana", "alice", "Bob", "Eve", "frank"],
    " Revenue ($) ": ["1000", "2,500", "$3,000", "N/A", "1,000", "2500", "$4,500", "$0"],
    "DATE": pd.date_range("2026-01-01", periods=8, freq="W"),
    "Score": [85, 92, np.nan, 78, 85, 92, 88, 95],
})

st.write("**Messy Data:**")
st.dataframe(messy_df, use_container_width=True)

st.write("**Issues:** Column names with spaces, dollar signs in numbers, mixed case names, duplicate rows")

# Cleaning steps
cleaned = messy_df.copy()

# Step 1: Fix column names
cleaned.columns = cleaned.columns.str.strip().str.lower().str.replace(" ", "_").str.replace("($)", "")
st.write(f"**Step 1 — Columns:** {list(cleaned.columns)}")

# Step 2: Clean string columns
if "name" in cleaned.columns:
    cleaned["name"] = cleaned["name"].str.strip().str.title()

# Step 3: Parse numeric columns
if "revenue" in cleaned.columns:
    cleaned["revenue"] = (
        cleaned["revenue"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("N/A", "")
    )
    cleaned["revenue"] = pd.to_numeric(cleaned["revenue"], errors="coerce")

# Step 4: Remove duplicates
n_before = len(cleaned)
cleaned = cleaned.drop_duplicates()
n_dupes = n_before - len(cleaned)

st.write(f"**Step 4 — Removed {n_dupes} duplicates**")
st.dataframe(cleaned, use_container_width=True)

---

## 🔧 Build It: Download Buttons

Export processed data in multiple formats.

In [ ]:
st.header("Download Processed Data")

# Use the cleaned data from above
if "cleaned" in dir() and cleaned is not None and len(cleaned) > 0:
    dl_col1, dl_col2, dl_col3 = st.columns(3)

    with dl_col1:
        st.download_button(
            label="📥 Download CSV",
            data=cleaned.to_csv(index=False),
            file_name="cleaned_data.csv",
            mime="text/csv",
            use_container_width=True,
        )

    with dl_col2:
        import io
        buffer = io.BytesIO()
        with pd.ExcelWriter(buffer, engine="openpyxl") as writer:
            cleaned.to_excel(writer, index=False, sheet_name="Cleaned")
        st.download_button(
            label="📥 Download Excel",
            data=buffer.getvalue(),
            file_name="cleaned_data.xlsx",
            mime="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
            use_container_width=True,
        )

    with dl_col3:
        st.download_button(
            label="📥 Download JSON",
            data=cleaned.to_json(orient="records", indent=2, date_format="iso"),
            file_name="cleaned_data.json",
            mime="application/json",
            use_container_width=True,
        )

    st.caption("Click any button to download the cleaned data in that format.")
else:
    st.info("Upload and clean data above to enable downloads.")

---

## 🧪 Experiment: Full Pipeline

Let's put it all together: upload → validate → clean → analyze → download.

In [ ]:
st.header("🧪 Full Pipeline Demo")

# Upload
uploaded_full = st.file_uploader(
    "Upload CSV for full pipeline",
    type=["csv"],
    key="full_pipeline",
)

if uploaded_full is not None:
    # Validate & Read
    try:
        uploaded_full.seek(0)
        df_full = pd.read_csv(uploaded_full)
    except Exception as e:
        st.error(f"Failed to read file: {e}")
        st.stop()

    # Data quality summary
    st.subheader("Step 1: Data Summary")
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Rows", f"{len(df_full):,}")
    c2.metric("Columns", len(df_full.columns))
    c3.metric("Null Cells", f"{df_full.isnull().sum().sum():,}")
    c4.metric("Memory", f"{df_full.memory_usage(deep=True).sum() / 1024:.0f} KB")

    # Clean options
    st.subheader("Step 2: Clean Options")
    clean_col1, clean_col2 = st.columns(2)
    with clean_col1:
        drop_dupes = st.checkbox("Remove duplicate rows", value=True)
        fix_names = st.checkbox("Standardize column names", value=True)
    with clean_col2:
        null_strategy = st.selectbox(
            "Handle missing values",
            ["Keep as-is", "Drop rows", "Fill with mean", "Fill with median"],
            key="pipeline_null",
        )

    # Apply cleaning
    df_clean_full = df_full.copy()

    if fix_names:
        df_clean_full.columns = df_clean_full.columns.str.strip().str.lower().str.replace(" ", "_")

    if drop_dupes:
        df_clean_full = df_clean_full.drop_duplicates()

    if null_strategy == "Drop rows":
        df_clean_full = df_clean_full.dropna()
    elif null_strategy == "Fill with mean":
        num_cols = df_clean_full.select_dtypes(include="number").columns
        df_clean_full[num_cols] = df_clean_full[num_cols].fillna(df_clean_full[num_cols].mean())
    elif null_strategy == "Fill with median":
        num_cols = df_clean_full.select_dtypes(include="number").columns
        df_clean_full[num_cols] = df_clean_full[num_cols].fillna(df_clean_full[num_cols].median())

    # Show before/after
    st.subheader("Step 3: Results")
    tab_before, tab_after = st.tabs(["Before Cleaning", "After Cleaning"])
    with tab_before:
        st.dataframe(df_full.head(20), use_container_width=True, height=300)
    with tab_after:
        st.dataframe(df_clean_full.head(20), use_container_width=True, height=300)

    st.write(f"Rows: {len(df_full):,} → {len(df_clean_full):,} | Nulls: {df_full.isnull().sum().sum():,} → {df_clean_full.isnull().sum().sum():,}")

    # Download
    st.subheader("Step 4: Download")
    st.download_button(
        "📥 Download Cleaned CSV",
        data=df_clean_full.to_csv(index=False),
        file_name="cleaned_pipeline.csv",
        mime="text/csv",
        type="primary",
    )

---

## ⚠️ Common Mistakes

### Mistake 1: Not Handling None

```python
# ❌ Crashes if no file uploaded
df = pd.read_csv(uploaded)  # TypeError if uploaded is None

# ✅ Always check first
if uploaded is not None:
    df = pd.read_csv(uploaded)
```

### Mistake 2: Reading File Multiple Times

```python
# ❌ File position advances
pd.read_csv(uploaded)   # Reads OK
pd.read_csv(uploaded)   # Returns empty!

# ✅ Reset position or read once
uploaded.seek(0)
df = pd.read_csv(uploaded)
```

### Mistake 3: No Error Handling

```python
# ❌ Crashes on bad files
df = pd.read_csv(uploaded)

# ✅ Graceful handling
try:
    df = pd.read_csv(uploaded)
except pd.errors.EmptyDataError:
    st.error("File is empty.")
except pd.errors.ParserError:
    st.error("Could not parse CSV.")
```

### Mistake 4: Rendering Raw HTML

```python
# ❌ XSS vulnerability
st.markdown(df.to_html(), unsafe_allow_html=True)

# ✅ Safe
st.dataframe(df)
```

---

## 🔍 Debugging Tips

| Symptom | Likely Cause | Fix |
|---|---|---|
| `TypeError: expected str, bytes or os.PathLike object` | Passing UploadedFile to path-based API | Use `pd.read_csv(uploaded)` directly |
| `EmptyDataError` | File has no data rows | Check file content, add error handling |
| `ParserError` | Wrong delimiter or encoding | Try `sep="\t"` or `encoding="latin-1"` |
| `UnicodeDecodeError` | Non-UTF-8 encoding | Try `encoding="latin-1"` or `encoding="cp1252"` |
| Second `read_csv` returns empty | File position at end | Call `uploaded.seek(0)` first |
| File not appearing in session | Missing `key` parameter | Add unique `key` to `file_uploader` |
| Download button not working | Data not in expected format | Ensure data is str, bytes, or file-like |
| Excel read fails | Missing openpyxl | `pip install openpyxl` |

---

## ✅ Best Practices

1. **Always check `uploaded is not None`** before processing.
2. **Always wrap parsing in try/except** — files can be malformed.
3. **Validate file type AND content** — extensions can be faked.
4. **Enforce size limits** — use `max_upload_size` per widget.
5. **Read the file once**, store in a variable — don't re-read.
6. **Show a data quality summary** before cleaning.
7. **Let users choose** their cleaning strategy.
8. **Provide before/after comparison** — builds trust.
9. **Always offer a download** — users want their cleaned data back.
10. **Sanitize filenames** if writing uploads to disk.

---

## ✏️ Exercises

### Exercise 1: Multi-Format Uploader
Build an app that accepts CSV, Excel, and JSON files. For each format, display:
- File name and size
- Number of rows and columns
- First 10 rows
- Column data types

### Exercise 2: Data Quality Reporter
Upload a CSV and generate a comprehensive quality report:
- Null percentage per column
- Duplicate row count
- Numeric column statistics
- Categorical column value counts
Display results in a tabbed layout.

### Exercise 3: Custom Cleaning Pipeline
Build a cleaning app with checkboxes for:
- Remove duplicates
- Standardize column names
- Convert data types
- Remove outliers (IQR method)
Show before/after comparison and provide download.

## 🚀 Challenge Problem

Build a **Complete Data Processing Dashboard** that:
1. Accepts CSV, Excel, or JSON uploads
2. Validates file type, size, and content
3. Shows a data quality summary (nulls, duplicates, types)
4. Provides cleaning options (checkboxes/selectbox)
5. Shows before/after comparison
6. Displays basic statistics and visualizations
7. Offers CSV, Excel, and JSON downloads
8. Handles all errors gracefully

Use the UPLOAD → VALIDATE → READ → CLEAN → ANALYZE → VISUALIZE → DOWNLOAD pipeline.

---

## 📌 Key Takeaways

1. **`st.file_uploader`** returns a BytesIO object — pass directly to `pd.read_csv()`.
2. **Always validate** — type, size, columns, data quality.
3. **Handle missing values** — let users choose the strategy.
4. **`st.download_button`** accepts strings, bytes, or file-like objects.
5. **Security matters** — sanitize filenames, enforce size limits, never render raw HTML.
6. **Read once, use many times** — don't re-read the uploaded file.
7. **Graceful errors** — always wrap parsing in try/except.

---

## 📚 Further Reading

- [st.file_uploader API Reference](https://docs.streamlit.io/develop/api-reference/widgets/st.file_uploader)
- [st.download_button API Reference](https://docs.streamlit.io/develop/api-reference/widgets/st.download_button)
- [Streamlit File Upload Guide](https://docs.streamlit.io/develop/concepts/design/files-and-uploads)
- [OWASP File Upload Cheat Sheet](https://cheatsheetseries.owasp.org/cheatsheets/File_Upload_Cheat_Sheet.html)

---

## 🔗 Related Materials

- 📖 Reading: [09 — File Upload, Validation & Processing](../readings/09_file_upload_and_processing.md)
- ✏️ Exercise: [09 — File Upload Workshop](../exercises/09_file_upload_workshop.py)
- 🖥️ Demo App: [09 — File Upload Demo](../apps/09_file_upload_demo.py)
- 📝 Quiz: [06 — File Upload & Processing](../quizzes/06_file_upload.md)